Notebook destiné aux tests d'exploration de données des comptes de dépenses des communes entre 2017 et 2023

Ci-dessous le code d'une extraction de données des 100 premières lignes des comptes du département Loire-Atlantique via une API créant un fichier csv avec ces données.

In [21]:
import csv
import requests

URL = "https://data.ofgl.fr/api/explore/v2.1/catalog/datasets/ofgl-base-communes/records?where=dep_name%20%3D%20'Loire-Atlantique'"
response = requests.get(URL + "&limit=100&offset=0")
response.raise_for_status()
data1 = response.json().get("results", [])


with open('../data/raw/communes_loire-atlantique_2023.csv', 'w', newline='') as csvfile:
    spamwriter = csv.writer(csvfile, 
                            delimiter=';',
                            quotechar='|', 
                            quoting=csv.QUOTE_MINIMAL)
    
    spamwriter.writerow(data1[0].keys())
    
    for i in range(len(data1)):
        spamwriter.writerow(data1[i].values())

Affichage des informations principales des données extraites

Noms et quantité des colonnes et des lignes

In [22]:
import pandas as pd

# Affiche toutes les lignes
pd.set_option('display.max_rows', None)

# Affiche toutes les colonnes
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)         
# Évite de tronquer les valeurs longues dans une cellule
pd.set_option('display.max_colwidth', None)

df = pd.read_csv('../data/raw/communes_loire-atlantique_2023.csv', delimiter=';', encoding='ISO 8859-1')
df.head()

print(df.columns)
print(df.index)
print(df.shape)




Index(['exer', 'outre_mer', 'reg_code', 'reg_name', 'dep_code', 'dep_name',
       'epci_code', 'epci_name', 'tranche_population', 'rural', 'montagne',
       'touristique', 'tranche_revenu_imposable_par_habitant', 'qpv',
       'com_code', 'com_name', 'categ', 'siren', 'insee', 'ident', 'lbudg',
       'type_de_budget', 'nomen', 'agregat', 'montant', 'montant_en_millions',
       'ptot', 'euros_par_habitant', 'presence_budget', 'cbudg',
       'ordre_analyse1_section1', 'ordre_analyse1_section2',
       'ordre_analyse1_section3', 'ordre_analyse2_section1',
       'ordre_analyse2_section2', 'ordre_analyse2_section3',
       'ordre_analyse3_section1', 'ordre_analyse3_section2',
       'ordre_analyse3_section3', 'ordre_analyse4_section1', 'annee_join',
       'ptot_n'],
      dtype='object')
RangeIndex(start=0, stop=100, step=1)
(100, 42)


Noms des colonnes ayant des valeurs nulles

In [12]:
for column in df.columns:
    if(df[column].count() < 100):
        print(column)



ordre_analyse1_section1
ordre_analyse1_section2
ordre_analyse1_section3
ordre_analyse2_section1
ordre_analyse2_section2
ordre_analyse2_section3
ordre_analyse3_section1
ordre_analyse3_section2
ordre_analyse3_section3
ordre_analyse4_section1


Index des lignes dupliquées

In [13]:
print(df[df.duplicated()].index)

Index([], dtype='int64')


Extraction de la première ligne du DataFrame

In [16]:
print(df.iloc[0])

exer                                                                                     2017
outre_mer                                                                                 Non
reg_code                                                                                   52
reg_name                                                                     Pays de la Loire
dep_code                                                                                   44
dep_name                                                                     Loire-Atlantique
epci_code                                                                           200067346
epci_name                                Communauté d'agglomération Pornic Agglo Pays de Retz
tranche_population                                                                          6
rural                                                                                     Oui
montagne                                                    

Les catégories pertinentes à conserver sont :

Identification administrative
| Champ original | Nouveau nom      | Type | Description                                        |
| -------------- | ---------------- | ---- | -------------------------------------------------- |
| exer           | annee_exercice   | int  | Année d’exercice budgétaire                        |
| outre_mer      | zone_outre_mer   | bool | Indique si la collectivité est située en outre-mer |
| reg_name       | nom_region       | str  | Nom de la région                                   |
| dep_code       | code_departement | str  | Code du département                                |
| dep_name       | nom_departement  | str  | Nom du département                                 |
| com_code       | code_commune     | str  | Code INSEE de la commune                           |
| com_name       | nom_commune      | str  | Nom de la commune                                  |


Caractéristiques territoriales
| Champ original                        | Nouveau nom                 | Type | Description                                                               |
| ------------------------------------- | --------------------------- | ---- | ------------------------------------------------------------------------- |
| tranche_population                    | tranche_population_insee    | int  | Tranche de population selon la classification INSEE                       |
| rural                                 | commune_rurale              | bool | Indique si la commune est classée rurale                                  |
| montagne                              | zone_montagne               | bool | Indique si la commune se situe en zone de montagne                        |
| touristique                           | commune_touristique         | bool | Indique si la commune a un statut touristique particulier                 |
| tranche_revenu_imposable_par_habitant | tranche_revenu_par_habitant | int  | Niveau de revenu imposable moyen (1 = plus faible, n = plus élevé)        |
| qpv                                   | presence_qpv                | bool | Signale la présence de quartiers prioritaires de la politique de la ville |


Données budgétaires
| Champ original      | Nouveau nom          | Type  | Description                                              |
| ------------------- | -------------------- | ----- | -------------------------------------------------------- |
| lbudg               | libelle_budget       | str   | Libellé complet du budget                                |
| type_de_budget      | type_budget          | str   | Nature du budget (Budget principal, Budget annexe, etc.) |
| agregat             | poste_agrege         | str   | Niveau d’agrégation comptable                            |
| montant             | montant_euros        | float | Somme du poste budgétaire concerné (en euros)            |
| montant_en_millions | montant_millions     | float | Même montant exprimé en millions d’euros                 |
| ptot                | population_reference | int   | Population de référence pour l’exercice budgétaire       |
| euros_par_habitant  | euros_par_habitant   | float | Montant rapporté à la population (euros par habitant)    |

